In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

PyTorch: 2.12.1+cpu
Pandas: 2.2.3
NumPy: 2.5.0


In [2]:
df = pd.read_hdf("metr-la.h5")

print(df.shape)
df.head()

(34272, 207)


,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,62.750000,...,45.625000,65.500,64.500000,66.428571,66.875,59.375000,69.000000,59.250000,69.000000,61.875
2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444,68.111111,65.000,65.000,57.444444,63.333333,...,50.666667,69.875,66.666667,58.555556,62.000,61.111111,64.444444,55.888889,68.444444,62.875
2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857143,62.000
2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000
2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000


In [3]:
data = df.values

print(data.shape)

print("Min:", data.min())
print("Max:", data.max())
print("Mean:", data.mean())

(34272, 207)
Min: 0.0
Max: 70.0
Mean: 53.7190211024135


In [4]:
scaler = MinMaxScaler()

data_scaled = scaler.fit_transform(data)

print(data_scaled.shape)

(34272, 207)


In [5]:
SEQ_LEN = 12

X = []
y = []

for i in range(len(data_scaled) - SEQ_LEN):
    
    X.append(
        data_scaled[i:i+SEQ_LEN]
    )
    
    y.append(
        data_scaled[i+SEQ_LEN]
    )

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (34260, 12, 207)
y shape: (34260, 207)


In [6]:
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.1)

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[
    train_size:
    train_size+val_size
]

y_val = y[
    train_size:
    train_size+val_size
]

X_test = X[
    train_size+val_size:
]

y_test = y[
    train_size+val_size:
]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(23982, 12, 207)
(3426, 12, 207)
(6852, 12, 207)


In [7]:
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)

X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val)

X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

In [8]:
batch_size = 64

train_dataset = TensorDataset(
    X_train,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [9]:
class LSTMModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=207,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(
            128,
            207
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [10]:
model = LSTMModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

LSTMModel(
  (lstm): LSTM(207, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=207, bias=True)
)


In [11]:
epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(
            X_batch
        )

        loss = criterion(
            predictions,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (
        total_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch+1}/{epochs}"
        f" Loss: {avg_loss:.6f}"
    )

Epoch 1/30 Loss: 0.050480
Epoch 2/30 Loss: 0.032144
Epoch 3/30 Loss: 0.028868
Epoch 4/30 Loss: 0.027479
Epoch 5/30 Loss: 0.026330
Epoch 6/30 Loss: 0.025583
Epoch 7/30 Loss: 0.024809
Epoch 8/30 Loss: 0.023715
Epoch 9/30 Loss: 0.022902
Epoch 10/30 Loss: 0.022465
Epoch 11/30 Loss: 0.022148
Epoch 12/30 Loss: 0.022241
Epoch 13/30 Loss: 0.021607
Epoch 14/30 Loss: 0.021232
Epoch 15/30 Loss: 0.020933
Epoch 16/30 Loss: 0.020649
Epoch 17/30 Loss: 0.020421
Epoch 18/30 Loss: 0.020327
Epoch 19/30 Loss: 0.020187
Epoch 20/30 Loss: 0.020258
Epoch 21/30 Loss: 0.019821
Epoch 22/30 Loss: 0.019357
Epoch 23/30 Loss: 0.019192
Epoch 24/30 Loss: 0.019422
Epoch 25/30 Loss: 0.019050
Epoch 26/30 Loss: 0.018453
Epoch 27/30 Loss: 0.018158
Epoch 28/30 Loss: 0.017879
Epoch 29/30 Loss: 0.017602
Epoch 30/30 Loss: 0.017436


In [12]:
model.eval()

with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()
true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.09331757575273514
RMSE: 0.16512541233115222


In [13]:
import matplotlib.pyplot as plt

sensor = 0

plt.figure(figsize=(12,5))

plt.plot(
    true_values[:200, sensor],
    label="Actual"
)

plt.plot(
    predictions[:200, sensor],
    label="Predicted"
)

plt.legend()

plt.title(
    f"Sensor {sensor}"
)

plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["LSTM"],
    "MAE": [0.11330979],
    "RMSE": [0.20383552]
})

results.to_csv(
    "results.csv",
    index=False
)

print(results)

  Model      MAE      RMSE
0  LSTM  0.11331  0.203836
